In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
import torch
from torch.utils.data import Dataset
import torch.nn as nn
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, classification_report, log_loss

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATASET_PATH = Path("/content/drive/MyDrive/Colab Notebooks/tdcsfog")
TRAIN_PATH = DATASET_PATH / "train"
TEST_PATH = DATASET_PATH / "test"


In [28]:
file_path = '/content/drive/MyDrive/Colab Notebooks/tdcsfog/test/009ee11563_0000.npy'

with open(file_path, "rb") as infile:
        arr = np.load(infile)

print(arr.shape)
arr[1]
# # Extract the accelerometer data as the input features
# features = arr[:, 1:4]

# # Extract the labels
# labels = arr[:, 4:]
# labels = np.max(labels, axis=-1)
# return features, np.any(labels).astype(float)

(1280, 7)


array([ 1.        , -9.42509672,  0.7682455 , -1.75057992,  0.        ,
        0.        ,  0.        ])

In [56]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset

class NPYWindowDataset(Dataset):
    def __init__(self, folder_path):
        self.file_list = [
            os.path.join(folder_path, f)
            for f in os.listdir(folder_path) if f.endswith('.npy')
        ]

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        data = np.load(self.file_list[idx])             # shape: (1280, 7)
        X = torch.tensor(data[:, :4], dtype=torch.float32)  # shape: (1280, 4)

        labels = np.argmax(data[:, 4:], axis=1)         # shape: (1280,)
        label = 1 if 1 in labels else 0                 # label = 1 if any row == class 1
        y = torch.tensor(label, dtype=torch.float32)    # shape: scalar

        return X, y


In [58]:
import torch.nn as nn

class RNNModel(nn.Module):
    def __init__(self, input_size=4, hidden_size=64, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):  # x: (B, 1280, 4)
        rnn_out, _ = self.rnn(x)              # rnn_out: (B, 1280, hidden)
        last_hidden = rnn_out[:, -1, :]       # last timestep: (B, hidden)
        logits = self.fc(last_hidden).squeeze(-1)  # output: (B,)
        return logits  # raw logits (before sigmoid)


In [63]:
from torch.utils.data import DataLoader

train_dataset = NPYWindowDataset("/content/drive/MyDrive/Colab Notebooks/tdcsfog/train")
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

test_dataset = NPYWindowDataset("/content/drive/MyDrive/Colab Notebooks/tdcsfog/test")
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)

In [59]:


model = RNNModel().to("cuda")
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to("cuda"), y.to("cuda")
        optimizer.zero_grad()
        logits = model(X)                # (B,)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 175.3747
Epoch 2, Loss: 175.4128
Epoch 3, Loss: 175.2065
Epoch 4, Loss: 175.2918
Epoch 5, Loss: 174.8954
Epoch 6, Loss: 175.5532
Epoch 7, Loss: 175.4992
Epoch 8, Loss: 175.2530
Epoch 9, Loss: 175.3634
Epoch 10, Loss: 175.0458


In [65]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

y_true = []
y_pred = []
y_prob = []

model.eval()
with torch.no_grad():
    for X, y in test_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        logits = model(X)                    # (B,)
        probs = torch.sigmoid(logits)        # (B,)
        preds = (probs > 0.5).float()        # (B,)

        # Store results
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())   # for AUC

# ✅ Now compute metrics:
print("Accuracy       :", accuracy_score(y_true, y_pred))
print("Precision      :", precision_score(y_true, y_pred))
print("Recall         :", recall_score(y_true, y_pred))
print("F1 Score       :", f1_score(y_true, y_pred))
print("AUC-ROC        :", roc_auc_score(y_true, y_prob))
print("ConfusionMatrix:\n", confusion_matrix(y_true, y_pred))


Accuracy       : 0.6697916666666667
Precision      : 0.0
Recall         : 0.0
F1 Score       : 0.0
AUC-ROC        : 0.5
ConfusionMatrix:
 [[643   0]
 [317   0]]


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)
